In [26]:
# ── Cellule 1 : Imports + rechargement ───────────────────────────────────────
import pandas as pd
import numpy as np

PATH = "../data/raw/"

# Fichiers normaux
normal2023 = pd.read_csv(PATH + "envois_colis_2023.csv", sep=";", encoding='latin1')
normal2024 = pd.read_csv(PATH + "envois_colis_2024.csv", sep=";", encoding='latin1')
normal2025 = pd.read_csv(PATH + "envois_colis_2025.csv", sep=";", encoding='latin1')
normal2026 = pd.read_csv(PATH + "envois_colis_2026.csv", sep=";", encoding='latin1')

# Fichiers express
express2024 = pd.read_csv(PATH + "envois_express_2024.csv", sep=";", encoding='latin1')
express2025 = pd.read_csv(PATH + "envois_express_2025.csv", sep=";", encoding='latin1')
express2026 = pd.read_csv(PATH + "envois_express_2026.csv", sep=";", encoding='latin1')

fichiers = {
    "normal2023": normal2023, "normal2024": normal2024,
    "normal2025": normal2025, "normal2026": normal2026,
    "express2024": express2024, "express2025": express2025,
    "express2026": express2026,
}

print(" Fichiers chargés")
for nom, df in fichiers.items():
    print(f"   {nom} : {len(df):,} lignes")

C:\Users\eyato\AppData\Local\Temp\ipykernel_41456\2453226051.py:8: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  normal2023 = pd.read_csv(PATH + "envois_colis_2023.csv", sep=";", encoding='latin1')
C:\Users\eyato\AppData\Local\Temp\ipykernel_41456\2453226051.py:9: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  normal2024 = pd.read_csv(PATH + "envois_colis_2024.csv", sep=";", encoding='latin1')
C:\Users\eyato\AppData\Local\Temp\ipykernel_41456\2453226051.py:14: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  express2024 = pd.read_csv(PATH + "envois_express_2024.csv", sep=";", encoding='latin1')


 Fichiers chargés
   normal2023 : 50,653 lignes
   normal2024 : 129,005 lignes
   normal2025 : 144,330 lignes
   normal2026 : 62,275 lignes
   express2024 : 235,137 lignes
   express2025 : 342,798 lignes
   express2026 : 290,367 lignes


In [27]:
# ── Cellule 2 : Standardiser colonnes + convertir dates ──────────────────────

# Renommer NUMENVOI → NumEnvoi dans les fichiers express
for df in [express2024, express2025, express2026]:
    if "NUMENVOI" in df.columns:
        df.rename(columns={"NUMENVOI": "NumEnvoi"}, inplace=True)

# Convertir les dates
for nom, df in fichiers.items():
    df["Date"] = pd.to_datetime(df["Date"], format="%d/%m/%Y", errors="coerce")
    dates_invalides = df["Date"].isnull().sum()
    if dates_invalides > 0:
        print(f"  {nom} : {dates_invalides} dates invalides")
    else:
        print(f" {nom} : dates OK")

  normal2023 : 2 dates invalides
  normal2024 : 11 dates invalides
  normal2025 : 1 dates invalides
 normal2026 : dates OK
 express2024 : dates OK
 express2025 : dates OK
 express2026 : dates OK


In [28]:
# ── Cellule 3 : Analyser les types de doublons ───────────────────────────────

for nom, df in fichiers.items():
    doublons = df[df.duplicated(subset=["NumEnvoi"], keep=False)]
    if len(doublons) == 0:
        print(f"✅ {nom} : aucun doublon")
        continue

    print(f"\n{'='*50}")
    print(f" {nom}")

    # Type 1 : même NumEnvoi + même date + même poids + même montant
    type1 = doublons[
        doublons.duplicated(
            subset=["NumEnvoi", "Date", "Poids", "Montant"],
            keep=False
        )
    ]

    # Type 2 : même NumEnvoi mais données différentes
    type2 = doublons[
        ~doublons.duplicated(
            subset=["NumEnvoi", "Date", "Poids", "Montant"],
            keep=False
        )
    ]

    print(f"  Type 1 (vrais doublons — à supprimer) : {len(type1):,} lignes")
    print(f"  Type 2 (NumEnvoi réutilisé — à garder): {len(type2):,} lignes")


 normal2023
  Type 1 (vrais doublons — à supprimer) : 759 lignes
  Type 2 (NumEnvoi réutilisé — à garder): 236 lignes

 normal2024
  Type 1 (vrais doublons — à supprimer) : 2,730 lignes
  Type 2 (NumEnvoi réutilisé — à garder): 486 lignes

 normal2025
  Type 1 (vrais doublons — à supprimer) : 1,341 lignes
  Type 2 (NumEnvoi réutilisé — à garder): 286 lignes

 normal2026
  Type 1 (vrais doublons — à supprimer) : 898 lignes
  Type 2 (NumEnvoi réutilisé — à garder): 110 lignes

 express2024
  Type 1 (vrais doublons — à supprimer) : 2,419 lignes
  Type 2 (NumEnvoi réutilisé — à garder): 493 lignes

 express2025
  Type 1 (vrais doublons — à supprimer) : 1,983 lignes
  Type 2 (NumEnvoi réutilisé — à garder): 361 lignes

 express2026
  Type 1 (vrais doublons — à supprimer) : 559 lignes
  Type 2 (NumEnvoi réutilisé — à garder): 121 lignes


In [29]:
# ── Cellule 4 : Nettoyer les doublons ────────────────────────────────────────

def nettoyer_doublons(df, nom):
    avant = len(df)

    # Étape 1 — Supprimer Type 1
    df = df.drop_duplicates(
        subset=["NumEnvoi", "Date", "Poids", "Montant"],
        keep="first"
    )
    apres_type1 = len(df)

    # Étape 2 — Rendre Type 2 uniques avec suffixe
    masque_type2 = df.duplicated(subset=["NumEnvoi"], keep=False)
    nb_type2 = masque_type2.sum()

    if nb_type2 > 0:
        df = df.copy()
        compteur = df.groupby("NumEnvoi").cumcount() + 1
        df.loc[masque_type2, "NumEnvoi"] = (
            df.loc[masque_type2, "NumEnvoi"] + "_" +
            compteur[masque_type2].astype(str)
        )

    print(f" {nom} : {avant - apres_type1:,} doublons supprimés | {nb_type2:,} NumEnvoi renommés | {len(df):,} lignes restantes")
    return df

normal2023  = nettoyer_doublons(normal2023,  "normal2023")
normal2024  = nettoyer_doublons(normal2024,  "normal2024")
normal2025  = nettoyer_doublons(normal2025,  "normal2025")
normal2026  = nettoyer_doublons(normal2026,  "normal2026")
express2024 = nettoyer_doublons(express2024, "express2024")
express2025 = nettoyer_doublons(express2025, "express2025")
express2026 = nettoyer_doublons(express2026, "express2026")

fichiers = {
    "normal2023": normal2023, "normal2024": normal2024,
    "normal2025": normal2025, "normal2026": normal2026,
    "express2024": express2024, "express2025": express2025,
    "express2026": express2026,
}

 normal2023 : 454 doublons supprimés | 245 NumEnvoi renommés | 50,199 lignes restantes
 normal2024 : 1,830 doublons supprimés | 509 NumEnvoi renommés | 127,175 lignes restantes
 normal2025 : 818 doublons supprimés | 297 NumEnvoi renommés | 143,512 lignes restantes
 normal2026 : 701 doublons supprimés | 117 NumEnvoi renommés | 61,574 lignes restantes
 express2024 : 1,391 doublons supprimés | 509 NumEnvoi renommés | 233,746 lignes restantes
 express2025 : 1,118 doublons supprimés | 370 NumEnvoi renommés | 341,680 lignes restantes
 express2026 : 297 doublons supprimés | 122 NumEnvoi renommés | 290,070 lignes restantes


In [30]:
# ── Cellule 5 : Vérification qualité ─────────────────────────────────────────

def verifier_qualite(df, nom, type_fichier="normal"):
    print(f"\n{'='*55}")
    print(f" {nom} — {len(df):,} lignes")

    if type_fichier == "normal":
        cols_obligatoires = ["NumEnvoi", "IdBureau", "Date", "Poids", "Montant", "paysDest", "codepays"]
    else:
        cols_obligatoires = ["NumEnvoi", "IdBureau", "Date", "Poids", "Montant", "paysDest", "CodeISOPays", "CodeService"]

    cols_descriptives = ["cp", "cite", "ville", "cpDest", "citeDest", "villeDest"]

    print("\n Colonnes OBLIGATOIRES :")
    for col in cols_obligatoires:
        if col not in df.columns:
            print(f"   {col} :  colonne absente")
            continue
        nb = df[col].isnull().sum()
        status = f"  {nb:,} manquantes" if nb > 0 else "OK"
        print(f"   {col} : {status}")

    print("\n Colonnes DESCRIPTIVES :")
    for col in cols_descriptives:
        if col not in df.columns:
            continue
        nb = df[col].isnull().sum()
        status = f"  {nb:,} manquantes → remplir Inconnu" if nb > 0 else "OK"
        print(f"   {col} : {status}")

    if "cp" in df.columns:
        cp_invalide = df[df["cp"].astype(str).str.strip() == "0000000"].shape[0]
        if cp_invalide > 0:
            print(f"\n  cp = '0000000' : {cp_invalide:,} lignes → remplir Inconnu")

    print("\n Valeurs numériques invalides :")
    print(f"   Poids <= 0  : {(df['Poids'] <= 0).sum():,} lignes → supprimer")
    print(f"   Montant <= 0: {(df['Montant'] <= 0).sum():,} lignes → supprimer")
    print(f"   Poids > 30  : {(df['Poids'] > 30).sum():,} lignes → supprimer")

    dates_nulles = df["Date"].isnull().sum()
    print(f"\n Dates invalides : {dates_nulles:,} lignes → supprimer")

for nom, df in fichiers.items():
    type_f = "express" if "express" in nom else "normal"
    verifier_qualite(df, nom, type_f)


 normal2023 — 50,199 lignes

 Colonnes OBLIGATOIRES :
   NumEnvoi : OK
   IdBureau : OK
   Date :   2 manquantes
   Poids : OK
   Montant : OK
   paysDest : OK
   codepays : OK

 Colonnes DESCRIPTIVES :
   cp :   1 manquantes → remplir Inconnu
   cite :   25 manquantes → remplir Inconnu
   ville :   774 manquantes → remplir Inconnu
   cpDest :   1,491 manquantes → remplir Inconnu
   citeDest :   1,324 manquantes → remplir Inconnu
   villeDest :   1,458 manquantes → remplir Inconnu

 Valeurs numériques invalides :
   Poids <= 0  : 13 lignes → supprimer
   Montant <= 0: 4 lignes → supprimer
   Poids > 30  : 4 lignes → supprimer

 Dates invalides : 2 lignes → supprimer

 normal2024 — 127,175 lignes

 Colonnes OBLIGATOIRES :
   NumEnvoi : OK
   IdBureau : OK
   Date :   9 manquantes
   Poids : OK
   Montant : OK
   paysDest : OK
   codepays : OK

 Colonnes DESCRIPTIVES :
   cp : OK
   cite :   18 manquantes → remplir Inconnu
   ville :   1,023 manquantes → remplir Inconnu
   cpDest :   1,

In [31]:
# ── Cellule 6 : Nettoyage complet ────────────────────────────────────────────

def nettoyer_fichier(df, nom, type_fichier="normal"):
    avant = len(df)
    print(f"\n{'='*55}")
    print(f"📁 {nom} — {avant:,} lignes")

    # ── Supprimer lignes invalides ────────────────────────────────────────────
    df = df.dropna(subset=["Date"])
    df = df[df["Poids"] > 0]
    df = df[df["Poids"] <= 30]
    df = df[df["Montant"] > 0]

    if type_fichier == "express":
        df = df.dropna(subset=["CodeService"])
        df = df[df["CodeService"].isin(["EMS-N", "EMS-I", "RPP-I"])]

    apres_suppression = len(df)

    # ── Remplir colonnes descriptives ─────────────────────────────────────────
    cols_a_remplir = ["cp", "cite", "ville", "cpDest", "citeDest", "villeDest"]
    for col in cols_a_remplir:
        if col in df.columns:
            df[col] = df[col].fillna("Inconnu")

    if "cp" in df.columns:
        df["cp"] = df["cp"].astype(str).str.strip()
        df.loc[df["cp"] == "0000000", "cp"] = "Inconnu"

    # ── Normaliser strings ────────────────────────────────────────────────────
    cols_str = ["ville", "cite", "villeDest", "citeDest", "paysDest"]
    for col in cols_str:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().str.upper()

    # ── Ajouter colonnes calculées ────────────────────────────────────────────
    if type_fichier == "express":
        df["portee"] = df["CodeISOPays"].apply(
            lambda x: "National" if str(x).strip().upper() == "TN" else "International"
        )
        mapping_service = {
            "EMS-N": "Express normal",
            "EMS-I": "Express personnalisé",
            "RPP-I": "Express personnalisé",
        }
        df["type_service"] = df["CodeService"].map(mapping_service)
        df["CodeISOPays"] = df["CodeISOPays"].astype(str).str.strip().str.upper()
        if "RefPaiement" not in df.columns:
            df["RefPaiement"] = None
    else:
        df["portee"] = df["codepays"].apply(
            lambda x: "National" if str(x).strip().upper() == "TN" else "International"
        )
        df["CodeService"] = df["NumEnvoi"].str[:2].map({
            "CP": "CP", "UP": "UP", "RR": "RR",
        }).fillna("NOR")
        df["type_service"] = "Normal"
        df["CodeISOPays"] = df["codepays"].astype(str).str.strip().str.upper()

    df["type_source"] = type_fichier

    apres = len(df)
    print(f"   Lignes supprimées : {avant - apres_suppression:,}")
    print(f"   Lignes finales    : {apres:,}")
    print(f"   Total supprimé    : {avant - apres:,} ({round((avant - apres) / avant * 100, 1)}%)")

    return df

normal2023  = nettoyer_fichier(normal2023,  "normal2023",  "normal")
normal2024  = nettoyer_fichier(normal2024,  "normal2024",  "normal")
normal2025  = nettoyer_fichier(normal2025,  "normal2025",  "normal")
normal2026  = nettoyer_fichier(normal2026,  "normal2026",  "normal")
express2024 = nettoyer_fichier(express2024, "express2024", "express")
express2025 = nettoyer_fichier(express2025, "express2025", "express")
express2026 = nettoyer_fichier(express2026, "express2026", "express")

fichiers = {
    "normal2023": normal2023, "normal2024": normal2024,
    "normal2025": normal2025, "normal2026": normal2026,
    "express2024": express2024, "express2025": express2025,
    "express2026": express2026,
}

print("\n Nettoyage terminé")


📁 normal2023 — 50,199 lignes
   Lignes supprimées : 20
   Lignes finales    : 50,179
   Total supprimé    : 20 (0.0%)

📁 normal2024 — 127,175 lignes
   Lignes supprimées : 114
   Lignes finales    : 127,061
   Total supprimé    : 114 (0.1%)

📁 normal2025 — 143,512 lignes
   Lignes supprimées : 91
   Lignes finales    : 143,421
   Total supprimé    : 91 (0.1%)

📁 normal2026 — 61,574 lignes
   Lignes supprimées : 28
   Lignes finales    : 61,546
   Total supprimé    : 28 (0.0%)

📁 express2024 — 233,746 lignes
   Lignes supprimées : 4
   Lignes finales    : 233,742
   Total supprimé    : 4 (0.0%)

📁 express2025 — 341,680 lignes
   Lignes supprimées : 27
   Lignes finales    : 341,653
   Total supprimé    : 27 (0.0%)

📁 express2026 — 290,070 lignes
   Lignes supprimées : 12
   Lignes finales    : 290,058
   Total supprimé    : 12 (0.0%)

 Nettoyage terminé


In [32]:
# ── Vérification IdBureau ─────────────────────────────────────────────────────
import re

all_bureaux = set()
for nom, df in fichiers.items():
    all_bureaux.update(df["IdBureau"].astype(str).str.strip().unique())

# Séparer bureaux numériques et agences alphabétiques
bureaux_num   = sorted([b for b in all_bureaux if re.match(r"^\d+$", b)])
agences_alpha = sorted([b for b in all_bureaux if not re.match(r"^\d+$", b)])

print(f"Bureaux numériques  : {len(bureaux_num)}")
print(f"Agences alphabétiques : {len(agences_alpha)}")
print(f"\nListe complète des agences :")
for a in agences_alpha:
    print(f"  {a}")

print(f"\nExemples bureaux numériques :")
print(bureaux_num[:20])

Bureaux numériques  : 641
Agences alphabétiques : 37

Liste complète des agences :
  2x34
  AHAM
  AHCH
  AKAS
  AKEF
  BEJA
  BNZA
  GABA
  GAFA
  JENA
  JERA
  KAIA
  KEBA
  MAHA
  MEDA
  MONA
  NBLA
  SFXB
  SFXC
  SIDA
  SILA
  SSEA
  TATA
  TBKA
  TOZA
  TUND
  TUNE
  TUNH
  TUNI
  TUNK
  TUNM
  TUNN
  TUNO
  TUNP
  TUNR
  TUNU
  ZARA

Exemples bureaux numériques :
['1000', '1001', '1002', '1003', '1004', '1005', '1006', '1007', '1008', '1009', '1013', '1017', '1018', '1019', '1023', '1027', '1029', '1046', '1049', '1053']


In [33]:
# ── Mapping complet agences alphabétiques → ID numérique ─────────────────────
# Convention : IDs à partir de 9001
# Groupés par ville probable pour faciliter le mapping gouvernorat

MAPPING_AGENCES = {
    # Tunis et environs
    "TUND": 9001,   # Tunis antenne D
    "TUNE": 9002,   # Tunis antenne E
    "TUNH": 9003,   # Tunis antenne H
    "TUNI": 9004,   # Tunis antenne I
    "TUNK": 9005,   # Tunis antenne K
    "TUNM": 9006,   # Tunis antenne M
    "TUNN": 9007,   # Tunis antenne N
    "TUNO": 9008,   # Tunis antenne O
    "TUNP": 9009,   # Tunis antenne P
    "TUNR": 9010,   # Tunis antenne R
    "TUNU": 9011,   # Tunis antenne U
    "TUNB": 9012,   # Tunis antenne B (si présent)

    # Sfax
    "SFXB": 9013,   # Sfax antenne B
    "SFXC": 9014,   # Sfax antenne C

    # Béja
    "BEJA": 9015,

    # Bizerte
    "BNZA": 9016,

    # Gabès
    "GABA": 9017,
    "GAFA": 9018,

    # Jendouba
    "JENA": 9019,
    "JERA": 9020,

    # Kairouan
    "KAIA": 9021,
    "KEBA": 9022,

    # Mahdia
    "MAHA": 9023,
    "MEDA": 9024,

    # Monastir
    "MONA": 9025,

    # Nabeul
    "NBLA": 9026,

    # Sidi Bouzid
    "SIDA": 9027,
    "SILA": 9028,

    # Siliana
    "SSEA": 9029,

    # Tataouine
    "TATA": 9030,

    # Tobarka / Tabarka
    "TBKA": 9031,

    # Tozeur
    "TOZA": 9032,

    # Zaghouan
    "ZARA": 9033,

    # Autres
    "AHAM": 9034,
    "AHCH": 9035,
    "AKAS": 9036,
    "AKEF": 9037,

    # Code suspect
    "2x34": 9038,
}

# Mapping gouvernorat par agence
# (pour retrouver le gouvernorat même sans fichier référence)
GOUVERNORAT_AGENCES = {
    "TUND": "Tunis", "TUNE": "Tunis", "TUNH": "Tunis",
    "TUNI": "Tunis", "TUNK": "Tunis", "TUNM": "Tunis",
    "TUNN": "Tunis", "TUNO": "Tunis", "TUNP": "Tunis",
    "TUNR": "Tunis", "TUNU": "Tunis",
    "SFXB": "Sfax",  "SFXC": "Sfax",
    "BEJA": "Béja",
    "BNZA": "Bizerte",
    "GABA": "Gabès", "GAFA": "Gabès",
    "JENA": "Jendouba", "JERA": "Jendouba",
    "KAIA": "Kairouan", "KEBA": "Kairouan",
    "MAHA": "Mahdia", "MEDA": "Mahdia",
    "MONA": "Monastir",
    "NBLA": "Nabeul",
    "SIDA": "Sidi Bouzid", "SILA": "Sidi Bouzid",
    "SSEA": "Siliana",
    "TATA": "Tataouine",
    "TBKA": "Tabarka",
    "TOZA": "Tozeur",
    "ZARA": "Zaghouan",
    "AHAM": "Inconnu", "AHCH": "Inconnu",
    "AKAS": "Inconnu", "AKEF": "Inconnu",
    "2x34": "Inconnu",
}

print("✅ Mapping défini")
print(f"   Agences mappées : {len(MAPPING_AGENCES)}")

✅ Mapping défini
   Agences mappées : 38


In [34]:
# ── Appliquer le mapping sur tous les fichiers ────────────────────────────────

for nom, df in fichiers.items():
    avant = df["IdBureau"].astype(str).str.strip().copy()

    # Remplacer les abréviations
    df["IdBureau_original"] = df["IdBureau"].astype(str).str.strip()
    df["IdBureau"] = df["IdBureau"].astype(str).str.strip().replace(MAPPING_AGENCES)
    df["IdBureau"] = pd.to_numeric(df["IdBureau"], errors="coerce").astype("Int64")

    non_convertis = df["IdBureau"].isnull().sum()
    if non_convertis > 0:
        print(f"⚠️  {nom} : {non_convertis} IdBureau non convertis")
        print(df[df["IdBureau"].isnull()]["IdBureau_original"].unique())
    else:
        print(f"✅ {nom} : IdBureau uniformisé — {df['IdBureau'].nunique()} bureaux uniques")

print("\n✅ Uniformisation IdBureau terminée")

✅ normal2023 : IdBureau uniformisé — 516 bureaux uniques
✅ normal2024 : IdBureau uniformisé — 620 bureaux uniques
✅ normal2025 : IdBureau uniformisé — 609 bureaux uniques
✅ normal2026 : IdBureau uniformisé — 590 bureaux uniques
✅ express2024 : IdBureau uniformisé — 568 bureaux uniques
✅ express2025 : IdBureau uniformisé — 600 bureaux uniques
✅ express2026 : IdBureau uniformisé — 574 bureaux uniques

✅ Uniformisation IdBureau terminée


In [35]:
# ── Vérification finale ───────────────────────────────────────────────────────

print("=== Répartition bureaux vs agences dans les données ===\n")

for nom, df in fichiers.items():
    bureaux = df[df["IdBureau"] < 9000]["IdBureau"].nunique()
    agences = df[df["IdBureau"] >= 9000]["IdBureau"].nunique()
    total_colis_bureaux = (df["IdBureau"] < 9000).sum()
    total_colis_agences = (df["IdBureau"] >= 9000).sum()
    print(f"{nom} :")
    print(f"   Bureaux (< 9000) : {bureaux} bureaux | {total_colis_agences:,} colis")
    print(f"   Agences (>= 9000): {agences} agences | {total_colis_bureaux:,} colis")

=== Répartition bureaux vs agences dans les données ===

normal2023 :
   Bureaux (< 9000) : 490 bureaux | 2,226 colis
   Agences (>= 9000): 26 agences | 47,953 colis
normal2024 :
   Bureaux (< 9000) : 584 bureaux | 4,050 colis
   Agences (>= 9000): 36 agences | 123,011 colis
normal2025 :
   Bureaux (< 9000) : 573 bureaux | 5,117 colis
   Agences (>= 9000): 36 agences | 138,304 colis
normal2026 :
   Bureaux (< 9000) : 554 bureaux | 2,517 colis
   Agences (>= 9000): 36 agences | 59,029 colis
express2024 :
   Bureaux (< 9000) : 533 bureaux | 10,016 colis
   Agences (>= 9000): 35 agences | 223,726 colis
express2025 :
   Bureaux (< 9000) : 541 bureaux | 73,924 colis
   Agences (>= 9000): 59 agences | 267,729 colis
express2026 :
   Bureaux (< 9000) : 515 bureaux | 161,763 colis
   Agences (>= 9000): 59 agences | 128,295 colis


In [36]:
# ── Trouver les agences non mappées dans express2025 et express2026 ───────────
import re

for nom in ["express2025", "express2026"]:
    df = fichiers[nom]
    
    # Récupérer les IdBureau_original qui sont >= 9000
    # mais qui ne sont pas dans notre mapping connu
    agences_sup = df[df["IdBureau"] >= 9000]["IdBureau_original"].unique()
    
    # Parmi celles-ci, lesquelles sont alphabétiques (non mappées avant)
    nouvelles = [a for a in agences_sup 
                 if not re.match(r"^\d+$", str(a).strip()) 
                 and str(a).strip() not in MAPPING_AGENCES]
    
    print(f"\n{nom} — nouvelles agences non mappées :")
    for a in sorted(nouvelles):
        print(f"  '{a}'")


express2025 — nouvelles agences non mappées :

express2026 — nouvelles agences non mappées :


In [37]:
# ── Vérifier les bureaux numériques >= 9000 ───────────────────────────────────
for nom, df in fichiers.items():
    # Bureaux originalement numériques mais >= 9000
    bureaux_grands = df[
        (df["IdBureau"] >= 9000) &
        (df["IdBureau_original"].str.match(r"^\d+$"))
    ]["IdBureau_original"].unique()
    
    if len(bureaux_grands) > 0:
        print(f"\n{nom} — bureaux numériques >= 9000 :")
        print(sorted(bureaux_grands))


normal2023 — bureaux numériques >= 9000 :
['9010', '9012', '9013', '9014', '9021', '9023', '9030', '9031', '9032', '9040', '9053', '9060', '9070', '9080', '9114', '9117', '9120', '9121', '9122', '9125', '9132', '9140', '9170', '9171']

normal2024 — bureaux numériques >= 9000 :
['9000', '9010', '9012', '9013', '9014', '9021', '9023', '9029', '9030', '9031', '9032', '9033', '9040', '9053', '9060', '9070', '9080', '9100', '9110', '9112', '9113', '9114', '9115', '9117', '9120', '9121', '9122', '9125', '9132', '9136', '9140', '9150', '9170', '9171', '9180']

normal2025 — bureaux numériques >= 9000 :
['9000', '9010', '9012', '9013', '9014', '9021', '9022', '9023', '9029', '9030', '9031', '9032', '9033', '9040', '9053', '9060', '9070', '9080', '9100', '9110', '9112', '9113', '9114', '9115', '9117', '9120', '9121', '9122', '9125', '9132', '9136', '9140', '9150', '9170', '9171', '9180']

normal2026 — bureaux numériques >= 9000 :
['9000', '9010', '9012', '9013', '9014', '9021', '9022', '9023', 

In [38]:
# ── Corriger la distinction bureaux vs agences ────────────────────────────────

# Recalculer avec le bon critère : type ORIGINAL de l'IdBureau
for nom, df in fichiers.items():
    df["est_agence"] = ~df["IdBureau_original"].str.match(r"^\d+$")

print("=== Répartition correcte bureaux vs agences ===\n")
for nom, df in fichiers.items():
    nb_bureaux = (~df["est_agence"]).sum()
    nb_agences = df["est_agence"].sum()
    print(f"{nom} :")
    print(f"   Bureaux numériques : {nb_bureaux:,} colis")
    print(f"   Agences            : {nb_agences:,} colis")

=== Répartition correcte bureaux vs agences ===

normal2023 :
   Bureaux numériques : 49,097 colis
   Agences            : 1,082 colis
normal2024 :
   Bureaux numériques : 126,686 colis
   Agences            : 375 colis
normal2025 :
   Bureaux numériques : 143,084 colis
   Agences            : 337 colis
normal2026 :
   Bureaux numériques : 61,233 colis
   Agences            : 313 colis
express2024 :
   Bureaux numériques : 233,742 colis
   Agences            : 0 colis
express2025 :
   Bureaux numériques : 277,262 colis
   Agences            : 64,391 colis
express2026 :
   Bureaux numériques : 133,152 colis
   Agences            : 156,906 colis


In [39]:
# ── Changer les IDs des agences pour éviter les conflits ─────────────────────
# On va utiliser 99001 à 99037 au lieu de 9001 à 9037

MAPPING_AGENCES_CORRIGE = {
    "2x34": 99001, "AHAM": 99002, "AHCH": 99003,
    "AKAS": 99004, "AKEF": 99005, "BEJA": 99006,
    "BNZA": 99007, "GABA": 99008, "GAFA": 99009,
    "JENA": 99010, "JERA": 99011, "KAIA": 99012,
    "KEBA": 99013, "MAHA": 99014, "MEDA": 99015,
    "MONA": 99016, "NBLA": 99017, "SFXB": 99018,
    "SFXC": 99019, "SIDA": 99020, "SILA": 99021,
    "SSEA": 99022, "TATA": 99023, "TBKA": 99024,
    "TOZA": 99025, "TUND": 99026, "TUNE": 99027,
    "TUNH": 99028, "TUNI": 99029, "TUNK": 99030,
    "TUNM": 99031, "TUNN": 99032, "TUNO": 99033,
    "TUNP": 99034, "TUNR": 99035, "TUNU": 99036,
    "ZARA": 99037,
}

GOUVERNORAT_AGENCES_CORRIGE = {
    "TUND": "Tunis", "TUNE": "Tunis", "TUNH": "Tunis",
    "TUNI": "Tunis", "TUNK": "Tunis", "TUNM": "Tunis",
    "TUNN": "Tunis", "TUNO": "Tunis", "TUNP": "Tunis",
    "TUNR": "Tunis", "TUNU": "Tunis",
    "SFXB": "Sfax",  "SFXC": "Sfax",
    "BEJA": "Béja",  "BNZA": "Bizerte",
    "GABA": "Gabès", "GAFA": "Gabès",
    "JENA": "Jendouba", "JERA": "Jendouba",
    "KAIA": "Kairouan", "KEBA": "Kairouan",
    "MAHA": "Mahdia",   "MEDA": "Mahdia",
    "MONA": "Monastir", "NBLA": "Nabeul",
    "SIDA": "Sidi Bouzid", "SILA": "Sidi Bouzid",
    "SSEA": "Siliana",  "TATA": "Tataouine",
    "TBKA": "Tabarka",  "TOZA": "Tozeur",
    "ZARA": "Zaghouan",
    "AHAM": "Inconnu",  "AHCH": "Inconnu",
    "AKAS": "Inconnu",  "AKEF": "Inconnu",
    "2x34": "Inconnu",
}

# Réappliquer le mapping corrigé sur tous les fichiers
for nom, df in fichiers.items():
    # Repartir de l'ID original
    df["IdBureau"] = df["IdBureau_original"].copy()
    
    # Remplacer seulement les agences alphabétiques
    df["IdBureau"] = df["IdBureau"].replace(MAPPING_AGENCES_CORRIGE)
    
    # Convertir en entier
    df["IdBureau"] = pd.to_numeric(df["IdBureau"], errors="coerce").astype("Int64")
    
    non_convertis = df["IdBureau"].isnull().sum()
    if non_convertis > 0:
        print(f"⚠️  {nom} : {non_convertis} non convertis")
    else:
        print(f"✅ {nom} : OK")

print("\n✅ Mapping corrigé appliqué")

✅ normal2023 : OK
✅ normal2024 : OK
✅ normal2025 : OK
✅ normal2026 : OK
✅ express2024 : OK
✅ express2025 : OK
✅ express2026 : OK

✅ Mapping corrigé appliqué


In [40]:
# ── Vérification finale correcte ─────────────────────────────────────────────
print("=== Répartition finale ===\n")
for nom, df in fichiers.items():
    bureaux = (~df["est_agence"]).sum()
    agences = df["est_agence"].sum()
    print(f"{nom} :")
    print(f"   Bureaux : {bureaux:,} colis | {df[~df['est_agence']]['IdBureau'].nunique()} bureaux uniques")
    print(f"   Agences : {agences:,} colis | {df[df['est_agence']]['IdBureau'].nunique()} agences uniques")

=== Répartition finale ===

normal2023 :
   Bureaux : 49,097 colis | 514 bureaux uniques
   Agences : 1,082 colis | 2 agences uniques
normal2024 :
   Bureaux : 126,686 colis | 619 bureaux uniques
   Agences : 375 colis | 1 agences uniques
normal2025 :
   Bureaux : 143,084 colis | 609 bureaux uniques
   Agences : 337 colis | 1 agences uniques
normal2026 :
   Bureaux : 61,233 colis | 590 bureaux uniques
   Agences : 313 colis | 1 agences uniques
express2024 :
   Bureaux : 233,742 colis | 568 bureaux uniques
   Agences : 0 colis | 0 agences uniques
express2025 :
   Bureaux : 277,262 colis | 576 bureaux uniques
   Agences : 64,391 colis | 34 agences uniques
express2026 :
   Bureaux : 133,152 colis | 549 bureaux uniques
   Agences : 156,906 colis | 36 agences uniques


In [41]:
# ── Diagnostic express2024 ────────────────────────────────────────────────────

print("=== IdBureau_original dans express2024 ===")
print(f"Type de la colonne : {express2024['IdBureau_original'].dtype}")
print(f"\nQuelques valeurs :")
print(express2024["IdBureau_original"].head(10).tolist())

print(f"\nValeurs non numériques :")
non_num = express2024[
    ~express2024["IdBureau_original"].astype(str).str.match(r"^\d+$")
]["IdBureau_original"].unique()
print(non_num)

print(f"\nNombre de valeurs non numériques : {len(non_num)}")

# Vérifier aussi si est_agence a bien été calculé
print(f"\nest_agence True  : {express2024['est_agence'].sum()}")
print(f"est_agence False : {(~express2024['est_agence']).sum()}")

=== IdBureau_original dans express2024 ===
Type de la colonne : object

Quelques valeurs :
['7050', '1095', '1095', '8122', '4171', '4171', '4171', '4171', '4171', '4171']

Valeurs non numériques :
[]

Nombre de valeurs non numériques : 0

est_agence True  : 0
est_agence False : 233742


In [42]:
# ── Sauvegarder les fichiers nettoyés ─────────────────────────────────────────
import os

PATH_OUT = "../data/processed/"
os.makedirs(PATH_OUT, exist_ok=True)

total = 0
for nom, df in fichiers.items():
    chemin = PATH_OUT + f"{nom}_clean.csv"
    df.to_csv(chemin, index=False, encoding="utf-8")
    total += len(df)
    print(f"✅ {nom}_clean.csv — {len(df):,} lignes")

print(f"\n📊 Total : {total:,} lignes prêtes pour la transformation")

✅ normal2023_clean.csv — 50,179 lignes
✅ normal2024_clean.csv — 127,061 lignes
✅ normal2025_clean.csv — 143,421 lignes
✅ normal2026_clean.csv — 61,546 lignes
✅ express2024_clean.csv — 233,742 lignes
✅ express2025_clean.csv — 341,653 lignes
✅ express2026_clean.csv — 290,058 lignes

📊 Total : 1,247,660 lignes prêtes pour la transformation


In [43]:
normal2023.Heure

0        1520
1        1031
2        0819
3        0824
4        0841
         ... 
50647     902
50648     812
50649     825
50650     849
50651     856
Name: Heure, Length: 50179, dtype: object